<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module2_Labs/Lab8_QAOA_Portfolio_Optimization.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# QOS Lab 8 — QAOA for Portfolio Optimization
### Quantum Optimization and Simulation | Cleveland State University
**Instructor:** Prof. Chansu Yu | Washkewicz College of Engineering

---
## Learning Objectives
1. Understand the **portfolio optimization** problem and its connection to Max-Cut
2. Download **real stock data** and compute a correlation matrix
3. Build a **correlation graph** where edge weights encode asset correlation
4. Map portfolio selection to a **QAOA Max-Cut** problem
5. Run QAOA to identify a **diversified portfolio** (assets with low mutual correlation)
6. *(Optional)* Validate on IBM Quantum hardware

---
### 📖 The Portfolio Optimization Story

**The problem:** Given $n$ stocks, select $k$ of them to hold. You want the portfolio to be **diversified** — stocks that don't all move together. If your tech stocks crash, your healthcare stocks shouldn't crash at the same time.

**Diversification = low correlation between holdings.**

**The Max-Cut connection:**
- Build a graph where nodes = stocks and edge weight $(i,j)$ = correlation between stock $i$ and stock $j$
- A high-weight edge means the two stocks move similarly (bad for diversification)
- **Maximizing the weighted cut** = selecting two groups of stocks that are **as uncorrelated as possible**
- We then pick the group with the assets we prefer (e.g., highest Sharpe ratio)

**Stocks in this lab:** AAPL (Apple), MSFT (Microsoft), TSLA (Tesla), JPM (JPMorgan), JNJ (Johnson & Johnson)

---

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

# Install yfinance if needed
# !pip install yfinance -q
import yfinance as yf

simulator = AerSimulator()

# ── Stock universe ─────────────────────────────────────────────────────────────
TICKERS = ['AAPL', 'MSFT', 'TSLA', 'JPM', 'JNJ']
N_STOCKS = len(TICKERS)

print("Portfolio Optimization with QAOA")
print(f"Stocks: {TICKERS}")
print(f"Number of stocks: {N_STOCKS}")
print(f"Number of binary partition variables: {N_STOCKS}")
print(f"Total possible portfolios: 2^{N_STOCKS} = {2**N_STOCKS}")

---
## Part 1: Fetch Real Stock Data and Build the Correlation Graph

In [ ]:
# ── 1.1  Download 2 years of daily returns ────────────────────────────────────
print("Downloading stock data from Yahoo Finance...")

try:
    raw = yf.download(TICKERS, period='2y', auto_adjust=True, progress=False)
    prices = raw['Close']
    print(f"Downloaded {len(prices)} trading days of data")
    print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
    DATA_AVAILABLE = True
except Exception as e:
    print(f"Download failed: {e}")
    print("Using synthetic data instead.")
    DATA_AVAILABLE = False

In [ ]:
# ── 1.2  Synthetic fallback (if no internet) ─────────────────────────────────
if not DATA_AVAILABLE:
    import pandas as pd
    np.random.seed(2024)
    n_days = 500
    dates  = pd.date_range('2022-01-01', periods=n_days, freq='B')

    # Construct correlated returns
    # Tech stocks (AAPL, MSFT, TSLA) are correlated with each other
    # Finance/Healthcare (JPM, JNJ) are less correlated with tech
    market_factor = np.random.normal(0, 0.01, n_days)  # common market
    tech_factor   = np.random.normal(0, 0.01, n_days)  # tech sector

    returns_data = {
        'AAPL': 0.6*market_factor + 0.5*tech_factor + 0.01*np.random.normal(0,1,n_days),
        'MSFT': 0.6*market_factor + 0.5*tech_factor + 0.01*np.random.normal(0,1,n_days),
        'TSLA': 0.4*market_factor + 0.4*tech_factor + 0.02*np.random.normal(0,1,n_days),
        'JPM':  0.5*market_factor + 0.1*tech_factor + 0.01*np.random.normal(0,1,n_days),
        'JNJ':  0.3*market_factor - 0.1*tech_factor + 0.01*np.random.normal(0,1,n_days),
    }
    returns = pd.DataFrame(returns_data, index=dates)
    prices  = (1 + returns).cumprod() * 100  # synthetic price
    print("Using synthetic stock data.")
    DATA_AVAILABLE = True

In [ ]:
# ── 1.3  Compute daily returns and statistics ─────────────────────────────────
returns = prices.pct_change().dropna()

print("Annualized Return and Risk:")
print(f"{'Ticker':>8} | {'Ann. Return':>12} | {'Ann. Volatility':>16} | {'Sharpe (rf=2%)':>15}")
print("-" * 60)

annual_return = {}
annual_vol    = {}
sharpe        = {}

for ticker in TICKERS:
    r   = returns[ticker]
    mu  = r.mean() * 252          # annualize
    sig = r.std() * np.sqrt(252)
    sh  = (mu - 0.02) / sig
    annual_return[ticker] = mu
    annual_vol[ticker]    = sig
    sharpe[ticker]        = sh
    print(f"{ticker:>8} | {mu:>11.2%} | {sig:>15.2%} | {sh:>15.2f}")

In [ ]:
# ── 1.4  Compute correlation matrix ──────────────────────────────────────────
corr = returns.corr()

print("Correlation Matrix:")
print(corr.round(3))

# Visualize
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap='RdYlGn_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson correlation')
ax.set_xticks(range(N_STOCKS)); ax.set_xticklabels(TICKERS)
ax.set_yticks(range(N_STOCKS)); ax.set_yticklabels(TICKERS)
ax.set_title('Stock Return Correlation Matrix\n(Red=highly correlated, Green=uncorrelated/negative)')
for i in range(N_STOCKS):
    for j in range(N_STOCKS):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color='black', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 1.5  Build the Correlation Graph ─────────────────────────────────────────
# Threshold: only connect stocks if correlation > threshold
# (We don't need an edge for uncorrelated stocks — no diversification gain to cut)
CORR_THRESHOLD = 0.3

G_portfolio = nx.Graph()
G_portfolio.add_nodes_from(range(N_STOCKS))

edges_portfolio = []
edge_weights    = []

for i in range(N_STOCKS):
    for j in range(i+1, N_STOCKS):
        c = abs(corr.iloc[i, j])  # use absolute correlation
        if c > CORR_THRESHOLD:
            G_portfolio.add_edge(i, j, weight=c)
            edges_portfolio.append((i, j))
            edge_weights.append(c)

print(f"Correlation graph edges (threshold={CORR_THRESHOLD}):")
for (i,j), w in zip(edges_portfolio, edge_weights):
    print(f"  {TICKERS[i]}—{TICKERS[j]}: correlation = {w:.3f}")

# Draw the graph
fig, ax = plt.subplots(figsize=(8, 5))
pos = nx.spring_layout(G_portfolio, seed=42)
node_colors = ['#4e79a7'] * N_STOCKS
edge_colors = [G_portfolio[u][v]['weight'] for u,v in G_portfolio.edges()]
weights_draw = [G_portfolio[u][v]['weight']*5 for u,v in G_portfolio.edges()]

labels_dict = {i: TICKERS[i] for i in range(N_STOCKS)}
nx.draw(G_portfolio, pos=pos, ax=ax, labels=labels_dict,
        node_color=node_colors, node_size=800,
        edge_color=edge_colors, edge_cmap=plt.cm.Reds,
        width=weights_draw, with_labels=True,
        font_size=11, font_color='white', font_weight='bold')
ax.set_title(f'Portfolio Correlation Graph (edges: |corr| > {CORR_THRESHOLD})\n'
             'Thicker/redder edge = more correlated stocks (harder to diversify)')
plt.tight_layout()
plt.show()

---
## Part 2: Map to QAOA Max-Cut and Run

In [ ]:
# ── 2.1  Weighted cut value function ──────────────────────────────────────────
# The weighted cut value = sum of edge weights for cut edges
# Higher weighted cut = more diversity between the two groups

def weighted_cut(bitstring, edges, weights):
    """Compute weighted cut value for portfolio diversification."""
    total = 0.0
    for (u,v), w in zip(edges, weights):
        if bitstring[u] != bitstring[v]:
            total += w
    return total

# ── 2.2  Enumerate all 32 possible portfolio partitions ───────────────────────
all_bitstrings = [format(i,'05b') for i in range(32)]
all_wcuts = [weighted_cut(bs, edges_portfolio, edge_weights) for bs in all_bitstrings]

# Find the best classical solution
best_wcut_idx = np.argmax(all_wcuts)
best_bs       = all_bitstrings[best_wcut_idx]
best_wcut     = all_wcuts[best_wcut_idx]

group0 = [TICKERS[i] for i in range(N_STOCKS) if best_bs[i]=='0']
group1 = [TICKERS[i] for i in range(N_STOCKS) if best_bs[i]=='1']

print(f"Classical brute-force solution:")
print(f"  Best partition: {best_bs}")
print(f"  Weighted cut:   {best_wcut:.4f}")
print(f"  Group 0 (buy):  {group0}")
print(f"  Group 1 (avoid): {group1}")
print("\nTop 5 partitions by diversification score:")
top5 = sorted(zip(all_bitstrings, all_wcuts), key=lambda x: x[1], reverse=True)[:5]
for bs, wc in top5:
    g0 = [TICKERS[i] for i in range(N_STOCKS) if bs[i]=='0']
    g1 = [TICKERS[i] for i in range(N_STOCKS) if bs[i]=='1']
    print(f"  {bs}  wcut={wc:.3f}  Group0={g0}  Group1={g1}")

In [ ]:
# ── 2.3  Build QAOA circuit for portfolio optimization ────────────────────────
def build_portfolio_qaoa_circuit(gammas, betas, n_stocks, edges, measure=True):
    """QAOA circuit — same structure as Max-Cut but with portfolio graph edges."""
    p  = len(gammas)
    qc = QuantumCircuit(n_stocks, n_stocks if measure else 0)
    qc.h(range(n_stocks))
    for layer in range(p):
        for u,v in edges:
            qc.cx(u,v); qc.rz(-gammas[layer],v); qc.cx(u,v)
        for i in range(n_stocks):
            qc.rx(2*betas[layer], i)
    if measure:
        qc.measure(range(n_stocks), range(n_stocks))
    return qc

# ── 2.4  Objective function (weighted cut) ────────────────────────────────────
def portfolio_F(params, p_layers, n_stocks, edges, weights):
    gammas = params[:p_layers]
    betas  = params[p_layers:]
    qc = build_portfolio_qaoa_circuit(gammas, betas, n_stocks, edges, measure=False)
    sv = Statevector(qc)
    return sum(
        weighted_cut(bs, edges, weights) * abs(amp)**2
        for bs, amp in zip(all_bitstrings, sv.data)
    )

# Preview the circuit
qc_demo = build_portfolio_qaoa_circuit([0.5], [0.3], N_STOCKS, edges_portfolio, measure=False)
print(f"Portfolio QAOA circuit (p=1): depth={qc_demo.depth()}, gates={qc_demo.count_ops()}")
print(qc_demo.draw('text'))

In [ ]:
# ── 2.5  Optimize QAOA parameters ────────────────────────────────────────────
p_layers = 1
history_portfolio = []

def obj_portfolio(params):
    F = portfolio_F(params, p_layers, N_STOCKS, edges_portfolio, edge_weights)
    history_portfolio.append(F)
    return -F

np.random.seed(42)
init_params = np.random.uniform(0, np.pi, 2*p_layers)

print("Optimizing QAOA parameters for portfolio Max-Cut...")
result = minimize(
    obj_portfolio, init_params,
    method='COBYLA',
    options={'maxiter': 200, 'rhobeg': 0.5}
)

best_params  = result.x
best_F_qaoa  = -result.fun
print(f"QAOA F* = {best_F_qaoa:.4f}  (classical optimal = {best_wcut:.4f})")
print(f"γ* = {best_params[0]:.4f} rad, β* = {best_params[1]:.4f} rad")

In [ ]:
# ── 2.6  Run and interpret final result ──────────────────────────────────────
gammas_opt = best_params[:p_layers]
betas_opt  = best_params[p_layers:]

qc_final = build_portfolio_qaoa_circuit(gammas_opt, betas_opt, N_STOCKS, edges_portfolio, measure=True)
result_final = simulator.run(transpile(qc_final, simulator), shots=3000).result()
counts_final = result_final.get_counts()

# Sort by count
top_results = sorted(counts_final.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top QAOA portfolio suggestions:")
print(f"{'Partition':>12} | {'Count':>6} | {'Prob':>6} | {'Div. Score':>12} | {'Portfolio':>35}")
print("-" * 80)

for bs, cnt in top_results:
    prob = cnt / 3000
    wc   = weighted_cut(bs, edges_portfolio, edge_weights)
    g0   = [TICKERS[i] for i in range(N_STOCKS) if bs[i]=='0']
    g1   = [TICKERS[i] for i in range(N_STOCKS) if bs[i]=='1']
    opt  = '← OPTIMAL' if abs(wc - best_wcut) < 0.001 else ''
    print(f"{bs:>12} | {cnt:>6} | {prob:>6.4f} | {wc:>12.4f} | {str(g0):>20} vs {str(g1):<15} {opt}")

In [ ]:
# ── 2.7  Visualization: QAOA measurement distribution ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: optimization convergence
axes[0].plot(history_portfolio, 'steelblue', lw=1.5, alpha=0.8)
axes[0].axhline(best_wcut, color='green', linestyle='--', lw=2, label=f'Classical optimal={best_wcut:.3f}')
mean_wcut = np.mean(all_wcuts)
axes[0].axhline(mean_wcut, color='gray', linestyle=':', lw=1.5, label=f'Random mean={mean_wcut:.3f}')
axes[0].set_xlabel('COBYLA evaluations'); axes[0].set_ylabel('Expected diversification score')
axes[0].set_title('QAOA Optimization for Portfolio Diversification')
axes[0].legend()

# Right: top measurement outcomes
top20 = sorted(counts_final.items(), key=lambda x: x[1], reverse=True)[:20]
top20_bs  = [t[0] for t in top20]
top20_cnt = [t[1] for t in top20]
top20_wc  = [weighted_cut(t[0], edges_portfolio, edge_weights) for t in top20]
bar_colors = plt.cm.RdYlGn([wc/best_wcut if best_wcut>0 else 0 for wc in top20_wc])

axes[1].bar(range(len(top20_bs)), top20_cnt, color=bar_colors)
axes[1].axhline(3000/32, color='blue', linestyle='--', lw=1.5, label='Random expectation')
axes[1].set_xticks(range(len(top20_bs)))
axes[1].set_xticklabels(top20_bs, rotation=90, fontsize=8)
axes[1].set_xlabel('Partition (bitstring)'); axes[1].set_ylabel('Measurement count')
axes[1].set_title('Top 20 QAOA Outcomes\n(Green=high diversification, Red=low)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Part 3: Interpreting the Result — What Should You Buy?

In [ ]:
# ── 3.1  Identify the best QAOA-suggested portfolio ───────────────────────────
# Sort by weighted cut value AND count
best_qaoa_partition = sorted(
    [(bs, cnt, weighted_cut(bs,edges_portfolio,edge_weights)) for bs,cnt in counts_final.items()],
    key=lambda x: x[2], reverse=True
)[0]

best_bs_qaoa = best_qaoa_partition[0]
best_wc_qaoa = best_qaoa_partition[2]

group0_qaoa  = [(TICKERS[i], sharpe[TICKERS[i]]) for i in range(N_STOCKS) if best_bs_qaoa[i]=='0']
group1_qaoa  = [(TICKERS[i], sharpe[TICKERS[i]]) for i in range(N_STOCKS) if best_bs_qaoa[i]=='1']

print("🏆 QAOA Portfolio Recommendation:")
print(f"   Partition: {best_bs_qaoa}")
print(f"   Diversification score: {best_wc_qaoa:.4f} (max possible: {best_wcut:.4f})")
print()
print("Group 0 (holdings candidate):")
for ticker, sh in sorted(group0_qaoa, key=lambda x:-x[1]):
    print(f"   {ticker}: Sharpe={sh:.2f}, Return={annual_return[ticker]:.1%}, Vol={annual_vol[ticker]:.1%}")

print("\nGroup 1 (alternative or hedge):")
for ticker, sh in sorted(group1_qaoa, key=lambda x:-x[1]):
    print(f"   {ticker}: Sharpe={sh:.2f}, Return={annual_return[ticker]:.1%}, Vol={annual_vol[ticker]:.1%}")

print()
print("Recommendation: hold the group with higher average Sharpe ratio.")
avg_sharpe_0 = np.mean([sharpe[t] for t,_ in group0_qaoa])
avg_sharpe_1 = np.mean([sharpe[t] for t,_ in group1_qaoa])
best_group   = group0_qaoa if avg_sharpe_0 >= avg_sharpe_1 else group1_qaoa
print(f"   Group 0 avg Sharpe: {avg_sharpe_0:.2f}")
print(f"   Group 1 avg Sharpe: {avg_sharpe_1:.2f}")
print(f"\n✅ Suggested portfolio: {[t for t,_ in best_group]}")

In [ ]:
# ── 3.2  Visualize the final portfolio split on the correlation graph ─────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graph visualization
ax = axes[0]
pos = nx.spring_layout(G_portfolio, seed=42)
node_colors = ['steelblue' if best_bs_qaoa[i]=='0' else 'tomato' for i in range(N_STOCKS)]
edge_colors_graph = [
    'red' if best_bs_qaoa[u]!=best_bs_qaoa[v] else 'lightgray'
    for u,v in G_portfolio.edges()
]
edge_widths_graph = [
    G_portfolio[u][v]['weight']*6 if best_bs_qaoa[u]!=best_bs_qaoa[v] else 1
    for u,v in G_portfolio.edges()
]
nx.draw(G_portfolio, pos=pos, ax=ax, labels=labels_dict,
        node_color=node_colors, node_size=900,
        edge_color=edge_colors_graph, width=edge_widths_graph,
        with_labels=True, font_size=11, font_color='white', font_weight='bold')
ax.set_title(f'QAOA Result: Partition {best_bs_qaoa}\nBlue=Group 0, Red=Group 1, Red edges=cut (diversity)')

# Bar chart: expected return and volatility for each group
ax2 = axes[1]
x = np.arange(N_STOCKS)
ax2.bar(x - 0.2, [annual_return[t]*100 for t in TICKERS], 0.4,
        color=node_colors, alpha=0.7, label='Ann. Return (%)', edgecolor='black')
ax2.bar(x + 0.2, [annual_vol[t]*100 for t in TICKERS], 0.4,
        color=node_colors, alpha=0.4, hatch='//', label='Ann. Volatility (%)', edgecolor='black')
ax2.set_xticks(x); ax2.set_xticklabels(TICKERS)
ax2.set_ylabel('%'); ax2.set_title('Return (solid) vs Volatility (hatched)\nBlue=Group 0, Red=Group 1')
ax2.legend()
ax2.axhline(0, color='black', lw=0.8)

plt.tight_layout()
plt.show()

---
### ✏️ Exercise 8.1 — Adjust the Correlation Threshold

The `CORR_THRESHOLD = 0.3` controls which edges are included in the graph.

1. Try **CORR_THRESHOLD = 0.1** (more edges) and **CORR_THRESHOLD = 0.5** (fewer edges)
2. How does the graph change? How many edges does each threshold produce?
3. Rerun QAOA for each threshold. Does the recommended portfolio change?
4. What are the trade-offs of using a higher vs lower threshold?

In [ ]:
# YOUR CODE HERE
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, threshold in zip(axes, [0.1, 0.3, 0.5]):
    edges_t, weights_t = [], []
    G_t = nx.Graph()
    G_t.add_nodes_from(range(N_STOCKS))

    for i in range(N_STOCKS):
        for j in range(i+1, N_STOCKS):
            c = abs(corr.iloc[i,j])
            if c > threshold:
                G_t.add_edge(i,j,weight=c)
                edges_t.append((i,j)); weights_t.append(c)

    if edges_t:
        # Quick QAOA optimization
        def obj_t(params, et=edges_t, wt=weights_t):
            F = portfolio_F(params, 1, N_STOCKS, et, wt)
            return -F
        np.random.seed(42)
        res_t = minimize(obj_t, np.random.uniform(0,np.pi,2), method='COBYLA',
                         options={'maxiter':100,'rhobeg':0.5})
        best_wc_t = -res_t.fun
        best_bs_t = max(all_bitstrings, key=lambda bs: weighted_cut(bs,edges_t,weights_t))
    else:
        best_bs_t = '00000'; best_wc_t = 0

    pos_t = nx.spring_layout(G_t, seed=42)
    node_c = ['steelblue' if best_bs_t[i]=='0' else 'tomato' for i in range(N_STOCKS)]
    nx.draw(G_t, pos=pos_t, ax=ax, labels=labels_dict,
            node_color=node_c, node_size=700,
            with_labels=True, font_size=9, font_color='white', font_weight='bold',
            edge_color='gray', width=2)
    g0_t = [TICKERS[i] for i in range(N_STOCKS) if best_bs_t[i]=='0']
    ax.set_title(f'Threshold={threshold}\n{len(edges_t)} edges\nPortfolio: {g0_t}')

plt.suptitle('QAOA Portfolio vs. Correlation Threshold', fontsize=13)
plt.tight_layout()
plt.show()

print("Trade-offs:")
print("  Low threshold (0.1):  more edges → richer problem, but more circuit gates")
print("  High threshold (0.5): fewer edges → simpler circuit, but less diversification info")
print("  In practice: choose threshold based on which correlations are financially meaningful")

---
## ⭐ Optional: Run on Real IBM Quantum Hardware

In [ ]:
USE_REAL_HARDWARE = False  # Set True if you have IBM Quantum access

if USE_REAL_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    service = QiskitRuntimeService(token="<Your IBM API Token>",
                               instance = "<Your IBM CRN #>",
                               channel="ibm_cloud")
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)
    print(f"Using backend: {backend.name}")

    # Use optimized parameters from simulator
    qc_hw = build_portfolio_qaoa_circuit(gammas_opt, betas_opt, N_STOCKS, edges_portfolio, measure=True)
    qc_hw_t = transpile(qc_hw, backend, optimization_level=3)
    print(f"Transpiled circuit depth: {qc_hw_t.depth()}")

    sampler  = Sampler(backend)
    hw_job   = sampler.run([qc_hw_t], shots=1000)
    print(f"Job submitted. ID: {hw_job.job_id()}")
    print("Waiting for results...")
    hw_result = hw_job.result()
    hw_counts = hw_result[0].data.c.get_counts()

    print("\nTop results from real hardware:")
    for bs, cnt in sorted(hw_counts.items(), key=lambda x:x[1], reverse=True)[:5]:
        wc = weighted_cut(bs, edges_portfolio, edge_weights)
        g0 = [TICKERS[i] for i in range(N_STOCKS) if bs[i]=='0']
        print(f"  {bs}: {cnt} shots, wcut={wc:.3f}, group={g0}")
else:
    print("Simulator mode. Set USE_REAL_HARDWARE=True for IBM Quantum.")

---
## ✅ Lab 8 Summary

### The Full QAOA-for-Finance Pipeline

| Step | What we did | Tool |
|------|-------------|------|
| 1. Data | Downloaded 2 years of daily returns for 5 stocks | `yfinance` |
| 2. Correlation | Computed return correlation matrix | `pandas` |
| 3. Graph | Built weighted correlation graph (edges = high correlation) | `networkx` |
| 4. QAOA | Mapped graph to Max-Cut; ran QAOA p=1 | Qiskit + Aer |
| 5. Result | Decoded measurement distribution → diversified portfolio partition | NumPy |
| 6. Decision | Chose the group with higher average Sharpe ratio | Finance |

### Key Connections

| Finance concept | Quantum concept |
|-----------------|----------------|
| Diversification | Max-Cut on correlation graph |
| Asset = node | Qubit |
| Correlation between assets | Edge weight |
| Portfolio partition | Bitstring |
| Maximizing diversity | Maximizing weighted cut |

---

## 🎓 Course Complete: What You Can Now Do

| Lab | Skill |
|-----|-------|
| Lab 1 | Python, NumPy, Qiskit basics |
| Lab 2 | Qubit states, Bloch sphere, Hadamard gate |
| Lab 3 | R_Z, R_X, Euler's formula, phase encoding |
| Lab 4 | Two-qubit gates, CNOT, R_ZZ, entanglement |
| Lab 5 | Max-Cut Hamiltonian, cost operator U_C |
| Lab 6 | Full QAOA circuit, multi-layer, real hardware |
| Lab 7 | COBYLA, SPSA, barren plateaus, initialization |
| Lab 8 | QAOA for portfolio optimization with real data |

---
*QOS Lab 8 | Prof. Chansu Yu | Cleveland State University*

> **Disclaimer:** This lab is for educational purposes only and does not constitute financial advice.